# GridGuard — Data audit and split design
Mahij Khantigiri GOSAI · ESILV Machine Learning

This notebook records the results of the locally executed audit on the supplied 1,000-row CSV and 19,000-row ZIP. The original 19,000-row archive must be supplied in `data/raw/` to rerun it. The data are simulated. No model is trained here.

In [1]:
from pathlib import Path
import json
import pandas as pd

root = Path.cwd()
if not (root / 'reports' / 'audit_metrics.json').exists():
    root = root.parent
metrics = json.loads((root / 'reports' / 'audit_metrics.json').read_text())
print('Records:', metrics['records'])
print('Input features:', metrics['input_features'])
print('Target counts:', metrics['target_counts'])
print('Positive rate: {:.2%}'.format(metrics['positive_rate']))

Records: 20000
Input features: 224
Target counts: {'0': 15968, '1': 4032}
Positive rate: 20.16%


## Source and scope
The original extracts are attributed to PowerGraph / IEEE-24 in the supplied source archive. `blackout=1` means at least some electricity demand is not supplied, not necessarily a nationwide blackout.

[PowerGraph dataset](https://figshare.com/articles/dataset/PowerGraph/22820534). The smaller extract lacks original scenario indices; feature timing remains unverified.

In [2]:
quality = {key: metrics[key] for key in (
    'missing_cells', 'nonfinite_input_cells', 'unique_feature_vectors',
    'repeated_feature_records', 'shared_feature_patterns_between_extracts',
    'conflicting_labels'
)}
for key, value in quality.items():
    print(f'{key}: {value}')

missing_cells: 0
nonfinite_input_cells: 0
unique_feature_vectors: 18579
repeated_feature_records: 1421
shared_feature_patterns_between_extracts: 152
conflicting_labels: 0


## Split strategy
Identical complete feature vectors are grouped before splitting so they cannot cross train, validation, and test. Target prevalence is used only to balance candidate partitions. These results are **not** the same protocol as the existing 1,000-row baseline notebook.

The source audit (`src/audit.py`) generates the authoritative manifests when both original extracts are available.

In [3]:
summary = pd.DataFrame(metrics['splits']).T
summary.index.name = 'partition'
summary

,records,positive
partition,,
train,13993,2820
validation,3010,607
test,2997,605


## Regenerate the outputs
Run this only after placing the original 19,000-row ZIP in `data/raw/` (the 1,000-row CSV already exists in `data/`). The audit recalculates the full summary and split manifests. It does **not** train a classifier.

```python
from src.audit import audit
fresh = audit(save=True)
```

For the exploratory charts and extended feature inventory, see the full locally executed notebook in the downloadable project package and `reports/data_audit.md`. Feature differences computed on all records are strictly descriptive, not training-time feature selection.

## What still needs checking
- Confirm whether each input is available before the cascade outcome.
- Identify scenario families beyond exact duplicate feature vectors.
- Verify raw feature definitions and any transformations.
- Do not tune preprocessing, feature selection, or thresholds on the test set.